# Benchmarking land-cover products for deforestation

The current deforestation analysis uses Hansen's yearly loss, which is not consistent over time: its method changed
in 2013 and the years before were never reprocessed (`hansen_annual_maps.ipynb`). Here Hansen is compared with the
other forest and land-cover products available on Earth Engine, before choosing what to use.

These products map what covers the ground as seen from space (trees, grass, crops, buildings, water), not what the land is used for. "Cropland" means *looks like* crops. 
Guinea's legal forest definition is partly about use (land used mainly for agriculture is excluded), so a land-cover map can only approximate it.

**Questions**
1. Which products exist, at what resolution, for which years, and are they made the same way every year? (sections 3–4)
2. Do they agree on where the forest is and on how much is lost each year? (section 8)
3. How do they compare with the official figures, NERF 2026 and FAO FRA 2020? (sections 8.2, 8.8, 8.9)
4. Which one is right? (section 10: needs reference points, not done yet)

## One Sample of Points

Every product is read at the same **systematic sample of points**: the centre of every [H3](https://h3geo.org/) hexagon of resolution 7 inside Guinea (~4.3 km² each, ~57,000 points, one every ~2.2 km).

- Each point stands for its hexagon, so **area of a class = sum of the hexagon areas of the points in that class**.
  This is the standard sample-based area estimate (Olofsson et al. 2014, *Remote Sensing of Environment* 148),
  with a standard error of ~0.2 percentage points of Guinea for a common class like forest. It is the same
  estimator Guinea's REDD+ reference level uses for its activity data (NERF 2026, following IPCC 2003 good
  practice guidance).
- Reading 57,000 points costs a small fraction of reading the whole map.
- Every product is read at exactly the same places and years, so they can be compared point by point.
- The same points are the population from which reference points are drawn for accuracy (section 10).

**Limit**: annual change is rare (≈ 0.5 % of the land per year, so ≈ 280 points), and its sample estimate has a relative error of ~6 %. Differences between products smaller than the error bars shown should not be read as real.

Each product is saved to its own parquet.

## 1. Setup

In [ ]:
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import h3
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().parent                     # .../Guinea_Methodology

KEY_CANDIDATES = [
    PROJECT_ROOT / "gee-account.json",
    PROJECT_ROOT / "Land Use-Deforestation" / "landuse_tool" / "credentials" / "tribal-mapper-275801-7b57ce84f3bd.json",
]
KEY_FILE = next((path for path in KEY_CANDIDATES if path.exists()), None)

OUTPUT_DIR = Path.cwd() / "output_datasets"
ECOREGIONS_GPKG = OUTPUT_DIR / "ecoregions_guinea.gpkg"            # Guinea boundary layer
ECOREGION_POLYGONS_GPKG = OUTPUT_DIR / "ecoregion_polygons.gpkg"   # one row per ecoregion polygon (ecoregions.ipynb)
HANSEN_LOSS_PARQUET = OUTPUT_DIR / "hansen_loss_by_polygon_cell_year.parquet"  # wall-to-wall Hansen (deforestation_hansen.ipynb)

BENCHMARK_DIR = OUTPUT_DIR / "lulc_benchmark"        # everything this notebook saves
BENCHMARK_DIR.mkdir(exist_ok=True)

c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.10.0). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)


## 2. Parameters

In [2]:
SAMPLE_H3_RESOLUTION = 7      # one point per H3 hexagon: 7 ≈ 4.3 km² (~57,000 points); 6 ≈ 30 km² (~8,000) for a quick try
MAP_H3_RESOLUTION = 5         # hexagons used to map disagreement (~250 km²)
COMPARISON_YEAR = 2020        # the year most products have (WorldCover, JRC GFC2020, GLAD, ...)
MAX_YEAR_GAP = 1              # a product without COMPARISON_YEAR uses its closest year within this gap
COMMON_WINDOW = (2017, 2022)  # years every annual product covers, for like-for-like time series
HANSEN_CANOPY_THRESHOLD = 30  # % canopy in 2000 that counts as forest, as in deforestation_hansen.ipynb
METHOD_CHANGE_YEAR = 2013     # Hansen method change (investigate_tier.ipynb)

CHUNK_SIZE = 4000                # points per Earth Engine request (a request can return at most 5,000 features)
MAX_EECU_HOURS_PER_PRODUCT = 10  # cost limit per product, see 6.2 (10 ≈ 7 % of the Community tier's 150 / month)
DYNAMIC_WORLD_YEARS = [COMPARISON_YEAR]  # ~8.6 EECU-hours per year; more years = FORCE_RESAMPLE and a higher limit
FORCE_RESAMPLE = False           # True = read the products on Earth Engine again even if the parquet exists
NODATA = -1                      # value given to masked pixels before sampling
SEED = 42

POINTS_PARQUET = BENCHMARK_DIR / f"sample_points_h3res{SAMPLE_H3_RESOLUTION}.parquet"
CATALOGUE_JSON = BENCHMARK_DIR / "catalogue.json"                     # assets and years found on Earth Engine
LABEL_TEMPLATE_CSV = BENCHMARK_DIR / f"points_to_label_{COMPARISON_YEAR}.csv"
LABEL_DESIGN_CSV = BENCHMARK_DIR / f"points_to_label_{COMPARISON_YEAR}_design.csv"  # strata, kept apart from interpreters
REFERENCE_CSV = BENCHMARK_DIR / f"reference_labels_{COMPARISON_YEAR}.csv"          # the template once labelled (section 10)

## 3. Products

| Key | Product | Provider | Resolution | Years | What counts as forest | Made the same way every year? |
|---|---|---|---|---|---|---|
| `hansen` | Global Forest Change v1.13 | UMD / GLAD | 30 m, Landsat | tree cover 2000 + loss 2001–2025 | canopy > 30 % in 2000, minus the loss | **No**: model changed in 2011/2013, earlier years not reprocessed |
| `tmf` | Tropical Moist Forest, annual change | EC JRC | 30 m, Landsat | 1990–2025, annual | undisturbed, degraded and regrowing **moist** forest | Yes: the whole archive is reprocessed at every release. Dry forest and savanna woodland are "other land" by design |
| `gfc2020` | Global Forest Cover 2020 (EUDR baseline) | EC JRC | 10 m | 2020 | FAO definition (> 10 % canopy, > 5 m, > 0.5 ha) | single year |
| `esri` | Annual Land Use / Land Cover | Esri / Impact Observatory | 10 m, Sentinel-2 | 2017 → | "Trees": tall, dense vegetation | same model, but each year classified on its own |
| `dynamic_world` | Dynamic World v1 | Google / WRI | 10 m, Sentinel-2 | mid-2015 → today, one map per image | "Trees" (here: the most frequent label of the year) | same model for every image; the annual map is our composite |
| `worldcover` | WorldCover | ESA | 10 m, Sentinel-1/2 | 2020 (v100), 2021 (v200) | "Tree cover" (> 10 % canopy) and mangroves | **No**: v200 uses a different algorithm; ESA warns that 2020→2021 differences are not change |
| `glc_fcs30d` | GLC_FCS30D | CAS (Zhang et al. 2024) | 30 m, Landsat | 2000–2022, annual | 10 forest types (open/closed, leaf type) and mangroves | yes: the series is built with change detection |
| `glad_glclu` | Global Land Cover and Land Use 2000–2020 | UMD / GLAD | 30 m, Landsat | 2000, 2005, 2010, 2015, 2020 | tree-height classes (trees defined by height, not canopy %) | yes: one Landsat analysis for all dates |
| `cgls_lc100` | Copernicus Global Land Cover | Copernicus / VITO | 100 m, PROBA-V | 2015–2019 | closed and open forest (> 15 % canopy) | yes, processed as a series |
| `modis` | MCD12Q1 v6.1, IGBP legend | NASA | 500 m | 2001 → | IGBP forests (> 60 % canopy) | yes, but a 500 m pixel is 25 ha: most clearings are smaller |

### One land-cover legend

- Each product has its own classes, so they are translated to a common **land-cover** legend: the seven classes every land-cover product here can express.
- They are the top-level classes of ESA WorldCover, whose legend is built with FAO's Land Cover Classification System (LCCS, Di Gregorio 2005); Esri and Dynamic World use almost the same classes. 
- Finer classes (forest types, crop types) are not shared by all products and are out of scope.
- These classes line up with the six IPCC land-use categories used in the NERF (forest land, cropland, grassland, wetlands, settlements, other land; IPCC 2006, Vol. 4 Ch. 3).
- Going from cover to use needs more information: a plantation or a cocoa / coffee agroforest is "Tree cover" here, and a fallow field is often "Shrub & grassland" while being part of the cropland.


| Code | Class |
|---|---|
| 1 | Tree cover (forest, mangroves) |
| 2 | Shrub & grassland (incl. savanna, rangeland) |
| 3 | Cropland |
| 4 | Built-up |
| 5 | Bare / sparse vegetation |
| 6 | Water |
| 7 | Herbaceous wetland |

`forest_classes` decides what counts as forest in each product; `crosswalk` translates its codes to the common legend.

In [3]:
HARMONISED_CLASSES = {
    1: "Tree cover", 2: "Shrub & grassland", 3: "Cropland", 4: "Built-up",
    5: "Bare / sparse vegetation", 6: "Water", 7: "Herbaceous wetland",
}
HARMONISED_COLORS = {
    "Tree cover": "#2e7d32", "Shrub & grassland": "#c5b358", "Cropland": "#f39c12", "Built-up": "#c0392b",
    "Bare / sparse vegetation": "#a1887f", "Water": "#2980b9", "Herbaceous wetland": "#7fcdbb",
}

PRODUCTS = {
    "hansen": {
        "label": "Hansen GFC", "kind": "forest", "scale": 30,
        "assets": ["UMD/hansen/global_forest_change_2025_v1_13"],
        # forest = canopy > HANSEN_CANOPY_THRESHOLD in 2000 and not lost by the end of the year (section 7)
    },
    "tmf": {
        "label": "JRC TMF", "kind": "forest", "scale": 30,
        "assets": ["projects/JRC/TMF/v1_2025/AnnualChanges", "projects/JRC/TMF/v1_2024/AnnualChanges",
                   "projects/JRC/TMF/v1_2023/AnnualChanges"],
        "forest_classes": [1, 2, 4],  # undisturbed, degraded, regrowing; 3 = deforested, 5 = water, 6 = other land
        "nodata": [],
    },
    "gfc2020": {
        "label": "JRC GFC2020", "kind": "forest", "scale": 10,
        "assets": ["JRC/GFC2020/V3", "JRC/GFC2020/V2", "JRC/GFC2020/V1"],
        "forest_classes": [1],        # only forest is mapped; everything else is masked and read as 0
        "nodata": [],
    },
    "esri": {
        "label": "Esri LULC", "kind": "lulc", "scale": 10, "band": "b1",
        "assets": ["projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS"],
        "forest_classes": [2],
        "nodata": [0, 9, 10],         # no data, snow, clouds
        "crosswalk": {1: 6, 2: 1, 4: 7, 5: 3, 7: 4, 8: 5, 11: 2},  # 11 = rangeland (grass + shrub)
    },
    "dynamic_world": {
        "label": "Dynamic World", "kind": "lulc", "scale": 10, "band": "label",
        "chunk_size": 1000,           # a year of per-image maps per point: the heaviest product to read
        "assets": ["GOOGLE/DYNAMICWORLD/V1"],
        "forest_classes": [1],
        "nodata": [8],                # snow
        "crosswalk": {0: 6, 1: 1, 2: 2, 3: 7, 4: 3, 5: 2, 6: 4, 7: 5},
    },
    "worldcover": {
        "label": "ESA WorldCover", "kind": "lulc", "scale": 10,
        "assets": {"2020": "ESA/WorldCover/v100", "2021": "ESA/WorldCover/v200"},
        "forest_classes": [10, 95],   # tree cover, mangroves
        "nodata": [0, 70],
        "crosswalk": {10: 1, 20: 2, 30: 2, 40: 3, 50: 4, 60: 5, 80: 6, 90: 7, 95: 1, 100: 2},
    },
    "glc_fcs30d": {
        "label": "GLC_FCS30D", "kind": "lulc", "scale": 30,
        "assets": ["projects/sat-io/open-datasets/GLC-FCS30D/annual"],
        "forest_classes": [51, 52, 61, 62, 71, 72, 81, 82, 91, 92, 185],  # open/closed forest types, mangroves
        "nodata": [0, 220, 250, 255],
        "crosswalk": {10: 3, 11: 3, 12: 3, 20: 3, 51: 1, 52: 1, 61: 1, 62: 1, 71: 1, 72: 1, 81: 1, 82: 1, 91: 1,
                      92: 1, 120: 2, 121: 2, 122: 2, 130: 2, 140: 2, 150: 5, 152: 5, 153: 5, 181: 7, 182: 7,
                      183: 7, 184: 7, 185: 1, 186: 7, 187: 7, 190: 4, 200: 5, 201: 5, 202: 5, 210: 6},
    },
    "glad_glclu": {
        "label": "GLAD GLCLU", "kind": "lulc", "scale": 30, "years": [2000, 2005, 2010, 2015, 2020],
        "assets": ["projects/glad/GLCLU2020/v2/LCLUC_{year}", "projects/glad/GLCLU2020/LCLUC_{year}"],
        "forest_classes": [*range(25, 49), *range(125, 149)],  # tree cover on dry land and in wetlands
        "nodata": [241, 254, 255],    # snow, ocean, no data
        "crosswalk": {0: 5, 1: 5, **{code: 2 for code in range(2, 25)}, **{code: 1 for code in range(25, 49)},
                      **{code: 7 for code in range(100, 125)}, **{code: 1 for code in range(125, 149)},
                      **{code: 6 for code in range(200, 208)}, 244: 3, 250: 4},
    },
    "cgls_lc100": {
        "label": "Copernicus LC100", "kind": "lulc", "scale": 100, "band": "discrete_classification",
        "assets": ["COPERNICUS/Landcover/100m/Proba-V-C3/Global"],
        "forest_classes": [*range(111, 117), *range(121, 127)],  # closed and open forest
        "nodata": [0, 70, 255],
        "crosswalk": {**{code: 1 for code in range(111, 117)}, **{code: 1 for code in range(121, 127)}, 20: 2,
                      30: 2, 40: 3, 50: 4, 60: 5, 80: 6, 90: 7, 100: 2, 200: 6},
    },
    "modis": {
        "label": "MODIS MCD12Q1", "kind": "lulc", "scale": 500, "band": "LC_Type1",
        "assets": ["MODIS/061/MCD12Q1"],
        # IGBP forests (> 60 % canopy). Class 8, woody savanna (30-60 % canopy), is mapped as shrub & grassland:
        # much of Guinea is class 8, so this choice changes MODIS forest area a lot.
        "forest_classes": [1, 2, 3, 4, 5],
        "nodata": [15, 255],
        "crosswalk": {1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 2, 7: 2, 8: 2, 9: 2, 10: 2, 11: 7, 12: 3, 13: 4, 14: 3,
                      16: 5, 17: 6},
    },
}
PRODUCT_LABELS = {key: spec["label"] for key, spec in PRODUCTS.items()}

# Hex colours: Earth Engine palettes do not accept the RGB strings plotly uses
PRODUCT_COLORS = {key: "#{:02x}{:02x}{:02x}".format(*map(int, px.colors.unlabel_rgb(color)))
                  for key, color in zip(PRODUCTS, px.colors.qualitative.Bold)}
LABEL_COLORS = {PRODUCT_LABELS[key]: color for key, color in PRODUCT_COLORS.items()}

## 4. Data Checking in Earth Engine

### 4.1 Connect

In [4]:
with open(KEY_FILE) as f:
    key_info = json.load(f)

SERVICE_ACCOUNT = key_info["client_email"]
GEE_PROJECT = key_info["project_id"]

credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, key_file=str(KEY_FILE))
ee.Initialize(credentials=credentials, project=GEE_PROJECT)

guinea_boundary = gpd.read_file(ECOREGIONS_GPKG, layer="guinea_boundary")
guinea_ee = ee.Geometry(guinea_boundary.geometry.iloc[0].simplify(0.001).__geo_interface__)  # ~100 m simplification
guinea_bbox = ee.Geometry.Rectangle(list(guinea_boundary.total_bounds))  # faster than the outline to filter images

### 4.2 Data Availability and Completeness Check

Each product lists candidate asset IDs, newest first, and the first one that exists is used. The years are read from the asset itself. Only metadata is read here and the result is saved to `catalogue.json`.

In [ ]:
def asset_type(asset_id):
    try:
        return ee.data.getAsset(asset_id)["type"]
    except ee.EEException:
        return None


def as_image(asset_id, kind):
    """The asset as one image: collections are mosaicked."""
    return ee.ImageCollection(asset_id).mosaic() if kind == "IMAGE_COLLECTION" else ee.Image(asset_id)


def discover(key):
    """Asset ID, asset type and years available for a product, or None if none of its assets exist."""
    spec = PRODUCTS[key]
    if key == "worldcover":
        years = [int(year) for year, asset in spec["assets"].items() if asset_type(asset)]
        return {"asset": spec["assets"], "asset_type": "IMAGE_COLLECTION", "years": years} if years else None
    if key == "glad_glclu":
        for template in spec["assets"]:
            kinds = {year: asset_type(template.format(year=year)) for year in spec["years"]}
            years = [year for year, kind in kinds.items() if kind]
            if years:
                return {"asset": template, "asset_type": kinds[years[0]], "years": years}
        return None

    for asset in spec["assets"]:
        kind = asset_type(asset)
        if kind:
            break
    else:
        return None

    if key == "hansen":
        years = list(range(2000, int(re.search(r"change_(\d{4})", asset).group(1)) + 1))
    elif key == "tmf":
        bands = as_image(asset, kind).bandNames().getInfo()
        years = sorted(int(band[3:]) for band in bands if re.fullmatch(r"Dec\d{4}", band) and int(band[3:]) >= 2000)
    elif key == "gfc2020":
        years = [2020]
    elif key == "glc_fcs30d":
        n_bands = as_image(asset, kind).bandNames().size().getInfo()
        years = list(range(2000, 2000 + n_bands))  # b1 = 2000, b2 = 2001, ...
    elif key == "dynamic_world":
        years = list(range(2016, datetime.now(timezone.utc).year))  # complete years only (2015 starts in June)
    else:
        millis = ee.ImageCollection(asset).filterBounds(guinea_bbox).aggregate_array("system:time_start").getInfo()
        years = sorted({datetime.fromtimestamp(m / 1000, timezone.utc).year for m in millis})
    return {"asset": asset, "asset_type": kind, "years": years}


if CATALOGUE_JSON.exists() and not FORCE_RESAMPLE:
    catalogue = json.loads(CATALOGUE_JSON.read_text())
    print(f"Read {CATALOGUE_JSON.name}, checked on {catalogue['checked_at_utc']}")
else:
    catalogue = {
        "checked_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "products": {key: discover(key) for key in PRODUCTS},
    }
    CATALOGUE_JSON.write_text(json.dumps(catalogue, indent=2))
    print(f"Checked Earth Engine and saved {CATALOGUE_JSON.name}")

available = {key: entry for key, entry in catalogue["products"].items() if entry and entry["years"]}
if "dynamic_world" in available:  # the most expensive product: only DYNAMIC_WORLD_YEARS are read
    dynamic_world = available["dynamic_world"]
    available["dynamic_world"] = {**dynamic_world, "years": [y for y in dynamic_world["years"] if y in DYNAMIC_WORLD_YEARS]}
pd.DataFrame(
    [
        {
            "product": PRODUCT_LABELS[key],
            "asset": json.dumps(entry["asset"]) if isinstance(entry["asset"], dict) else entry["asset"],
            "first_year": min(entry["years"]),
            "last_year": max(entry["years"]),
            "n_years": len(entry["years"]),
        }
        if entry and entry["years"] else {"product": PRODUCT_LABELS[key], "asset": "NOT FOUND"}
        for key, entry in catalogue["products"].items()
    ]
)

Read catalogue.json, checked on 2026-10-19T11:47:13+00:00


,product,asset,first_year,last_year,n_years
0,Hansen GFC,UMD/hansen/global_forest_change_2025_v1_13,2000,2025,26
1,JRC TMF,projects/JRC/TMF/v1_2025/AnnualChanges,2000,2025,26
2,JRC GFC2020,JRC/GFC2020/V3,2020,2020,1
3,Esri LULC,projects/sat-io/open-datasets/landcover/ESRI_G...,2017,2025,9
4,Dynamic World,GOOGLE/DYNAMICWORLD/V1,2016,2025,10
5,ESA WorldCover,"{""2020"": ""ESA/WorldCover/v100"", ""2021"": ""ESA/W...",2020,2021,2
6,GLC_FCS30D,projects/sat-io/open-datasets/GLC-FCS30D/annual,2000,2022,23
7,GLAD GLCLU,projects/glad/GLCLU2020/v2/LCLUC_{year},2000,2020,5
8,Copernicus LC100,COPERNICUS/Landcover/100m/Proba-V-C3/Global,2015,2019,5
9,MODIS MCD12Q1,MODIS/061/MCD12Q1,2001,2024,24


## 5. Ecoregions and sample points

The centre of every H3 hexagon (resolution `SAMPLE_H3_RESOLUTION`) that falls inside Guinea. 
The points are made locally and saved.

Each point carries:
- `weight_km2`: the area of its hexagon, the area the point stands for;
- `ECO_NAME`: its ecoregion (from `ecoregions.ipynb`);
- `h3_res5_id`: its parent hexagon, used for the maps.

The map below shows the ecoregions (from `ecoregions.ipynb`), with the same colours as `deforestation_hansen.ipynb`. Results are given per ecoregion in section 8.

In [6]:
polygons = gpd.read_file(ECOREGION_POLYGONS_GPKG).reset_index(names="polygon_index")
PARENT_COLUMN = f"h3_res{MAP_H3_RESOLUTION}_id"


def make_sample_points(resolution):
    cells = sorted(h3.h3shape_to_cells_experimental(h3.geo_to_h3shape(guinea_boundary.union_all()), resolution,
                                                    contain="center"))
    lats, lons = zip(*(h3.cell_to_latlng(cell) for cell in cells))
    sample = gpd.GeoDataFrame(
        {
            "point_id": range(len(cells)),  # cells are sorted, so neighbouring IDs are mostly neighbours on the ground
            "h3_id": cells,
            "lon": lons,
            "lat": lats,
            "weight_km2": [h3.cell_area(cell, unit="km^2") for cell in cells],
        },
        geometry=gpd.points_from_xy(lons, lats),
        crs="EPSG:4326",
    )
    sample = (
        sample.sjoin(polygons[["polygon_index", "ECO_NAME", "geometry"]], how="left", predicate="within")
        .drop(columns="index_right")
        .drop_duplicates("point_id")
    )
    sample["ECO_NAME"] = sample["ECO_NAME"].fillna("(outside the ecoregion polygons)")
    sample[PARENT_COLUMN] = sample["h3_id"].map(lambda cell: h3.cell_to_parent(cell, MAP_H3_RESOLUTION))
    return pd.DataFrame(sample.drop(columns="geometry")).reset_index(drop=True)


if POINTS_PARQUET.exists():
    points = pd.read_parquet(POINTS_PARQUET)
else:
    points = make_sample_points(SAMPLE_H3_RESOLUTION)
    points.to_parquet(POINTS_PARQUET, index=False)

weights = points.set_index("point_id")["weight_km2"]
TOTAL_KM2 = weights.sum()
guinea_km2 = guinea_boundary.to_crs(guinea_boundary.estimate_utm_crs()).area.sum() / 1e6
print(f"{len(points):,} points, {TOTAL_KM2:,.0f} km² represented (Guinea outline: {guinea_km2:,.0f} km²)")
points.groupby("ECO_NAME").agg(points=("point_id", "size"), area_km2=("weight_km2", "sum")).round(0)

56,712 points, 245,320 km² represented (Guinea outline: 244,632 km²)


,points,area_km2
ECO_NAME,,
(outside the ecoregion polygons),16,72.0
Guinean forest-savanna,31026,135438.0
Guinean mangroves,635,2895.0
Guinean montane forests,5986,26052.0
West Sudanian savanna,13674,59029.0
Western Guinean lowland forests,5375,21834.0


In [7]:
ecoregions = gpd.read_file(ECOREGIONS_GPKG, layer="ecoregions_guinea")
ecoregions["geometry"] = ecoregions.simplify(0.005)  # ~500 m, keeps the map light
ecoregions["sample_points"] = ecoregions["ECO_NAME"].map(points["ECO_NAME"].value_counts()).fillna(0).astype(int)
ecoregions["share_of_guinea_%"] = 100 * ecoregions["area_km2"] / ecoregions["area_km2"].sum()
ECOREGION_COLORS = dict(zip(sorted(ecoregions["ECO_NAME"]), px.colors.qualitative.Safe))  # as in deforestation_hansen.ipynb

fig = px.choropleth_map(
    ecoregions, geojson=ecoregions.set_index("ECO_NAME").__geo_interface__, locations="ECO_NAME", color="ECO_NAME",
    color_discrete_map=ECOREGION_COLORS,
    hover_data={"area_km2": ":,.0f", "share_of_guinea_%": ":.1f", "sample_points": ":,", "ECO_NAME": False},
    map_style="carto-positron", center={"lat": 10.4, "lon": -11}, zoom=5.6, opacity=0.7,
    labels={"ECO_NAME": "Ecoregion"}, title="Ecoregions of Guinea",
)
fig.update_layout(height=550, margin={"r": 0, "l": 0, "b": 0})
fig.show()

## 6. Read product at the points 

### 6.1 Image per product

For each product, one image with **one band per year** holding the product's own class codes is produced here. 

Exceptions:
- **Hansen**: the bands `treecover2000`, `lossyear` and `datamask`; the yearly forest map is computed locally.
- **Dynamic World** has one map per Sentinel-2 image; the annual map is the most frequent label of the year (`mode`).
- **JRC GFC2020** maps only forest: masked pixels are non-forest (0). 
- **JRC TMF** leaves land outside its maps masked; it is read as class 6, "other land".

Every other masked pixel is given the value `NODATA` (−1) before sampling.

In [8]:
def build_image(key, entry, years):
    """One band per year, named "2017", "2018", ..., holding the product's own class codes (Hansen: its bands)."""
    asset, kind = entry["asset"], entry["asset_type"]
    names = [str(year) for year in years]
    if key == "hansen":
        hansen = ee.Image(asset)
        return hansen.select(["treecover2000", "datamask"]).addBands(hansen.select("lossyear").unmask(0))
    if key == "tmf":
        return as_image(asset, kind).select([f"Dec{year}" for year in years], names).unmask(6)  # unmapped = other land
    if key == "gfc2020":
        return as_image(asset, kind).select([0], names).unmask(0)  # non-forest is masked
    if key == "worldcover":
        return ee.Image.cat([ee.ImageCollection(asset[str(year)]).first().select([0], [str(year)]) for year in years])
    if key == "glc_fcs30d":
        return as_image(asset, kind).select([f"b{year - 1999}" for year in years], names)
    if key == "glad_glclu":
        return ee.Image.cat([as_image(asset.format(year=year), kind).select([0], [str(year)]) for year in years])

    collection = ee.ImageCollection(asset).filterBounds(guinea_bbox).select([PRODUCTS[key]["band"]])
    annual = []
    for year in years:
        in_year = collection.filterDate(f"{year}-01-01", f"{year + 1}-01-01")
        annual.append((in_year.mode() if key == "dynamic_world" else in_year.mosaic()).rename(str(year)))
    return ee.Image.cat(annual)

### 6.2 Running Requests

The points are sent in chunks of `CHUNK_SIZE` (Dynamic World: 1,000).

**Cost limit.** A product whose projected cost is above `MAX_EECU_HOURS_PER_PRODUCT` is **not read** (a budget safeguard). The basis is the project's monthly compute allowance: a project on the Earth Engine Community tier gets 150 EECU-hours per month, shared by everyone using it
([Earth Engine noncommercial tiers](https://developers.google.com/earth-engine/guides/noncommercial_tiers)). 

The compute used by every product read is printed at the end of the cell.

The values of each product are saved to `lulc_benchmark/<product>_h3res<resolution>.parquet`, with the time and EECU-seconds used next to it in a `.run.json`.

In [27]:
def get_info_with_eecu(request):
    """Compute an Earth Engine object and return (result, seconds, eecu_seconds).
    eecu_seconds is None if the profile could not be fetched (it is sometimes not ready on the server yet).
    """
    profile_ids = []
    start = time.perf_counter()
    with ee.data.profiling(profile_ids.append):
        result = request.getInfo()
    seconds = time.perf_counter() - start

    get_profiles = ee.ApiFunction.lookup("Profile.getProfiles").call
    for attempt in range(5):
        try:
            profile_text = get_profiles(ids=profile_ids).getInfo()
        except ee.EEException:
            time.sleep(2 * (attempt + 1))
            continue
        # The profile is a table; its first column is EECU-seconds per operation ("-" when negligible).
        eecu_seconds = sum(
            float(line.split()[0]) for line in profile_text.splitlines()[1:] if line.split() and line.split()[0] != "-"
        )
        return result, seconds, round(eecu_seconds, 1)
    return result, seconds, None


def sample_chunk(image, chunk, scale):
    """Values of `image` at the points of `chunk`: (rows, seconds, eecu_seconds). A failing chunk is split in two."""
    features = [
        ee.Feature(ee.Geometry.Point([lon, lat]), {"point_id": int(point_id)})
        for point_id, lon, lat in zip(chunk["point_id"], chunk["lon"], chunk["lat"])
    ]
    request = image.sampleRegions(
        collection=ee.FeatureCollection(features), properties=["point_id"], scale=scale, geometries=False, tileScale=4,
    )
    try:
        result, seconds, eecu_seconds = get_info_with_eecu(request)
    except ee.EEException as error:
        if len(chunk) <= 250:
            raise
        print(f"    {len(chunk)} points failed ({str(error)[:80]}), retrying in two halves")
        half = len(chunk) // 2
        parts = [sample_chunk(image, chunk.iloc[:half], scale), sample_chunk(image, chunk.iloc[half:], scale)]
        return parts[0][0] + parts[1][0], parts[0][1] + parts[1][1], (parts[0][2] or 0) + (parts[1][2] or 0)
    return [feature["properties"] for feature in result["features"]], seconds, eecu_seconds


def sample_product(key, entry):
    """Read one product at all points: (table with one column per band, run info)."""
    spec = PRODUCTS[key]
    image = build_image(key, entry, entry["years"]).unmask(NODATA)
    size = spec.get("chunk_size", CHUNK_SIZE)
    chunks = [points.iloc[start:start + size] for start in range(0, len(points), size)]
    rows, seconds, eecu_seconds = [], 0.0, 0.0
    for number, chunk in enumerate(chunks, start=1):
        chunk_rows, chunk_seconds, chunk_eecu = sample_chunk(image, chunk, spec["scale"])
        rows += chunk_rows
        seconds += chunk_seconds
        eecu_seconds += chunk_eecu or 0
        print(f"  chunk {number}/{len(chunks)}: {chunk_seconds:.0f} s, {chunk_eecu} EECU-seconds")
        if number == 1 and chunk_eecu:
            projected_hours = chunk_eecu * len(chunks) / 3600
            print(f"  projected for the whole product: ~{projected_hours:.2f} EECU-hours")
            if projected_hours > MAX_EECU_HOURS_PER_PRODUCT:
                raise RuntimeError(
                    f"projected {projected_hours:.1f} EECU-hours > MAX_EECU_HOURS_PER_PRODUCT. Use fewer years, "
                    "a coarser SAMPLE_H3_RESOLUTION, or raise the limit."
                )
    table = pd.DataFrame(rows)
    value_columns = sorted(column for column in table.columns if column != "point_id")
    table[value_columns] = table[value_columns].round().astype("int64")
    run_info = {
        "computed_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "asset": entry["asset"],
        "years": entry["years"],
        "points": len(points),
        "points_returned": len(table),
        "h3_resolution": SAMPLE_H3_RESOLUTION,
        "seconds": round(seconds, 1),
        "eecu_seconds": round(eecu_seconds, 1),
        "eecu_hours": round(eecu_seconds / 3600, 3),
    }
    return table[["point_id", *value_columns]], run_info


def product_parquet(key):
    return BENCHMARK_DIR / f"{key}_h3res{SAMPLE_H3_RESOLUTION}.parquet"


sampled, stopped, failures = {}, {}, {}
for key, entry in available.items():
    path = product_parquet(key)
    if path.exists() and not FORCE_RESAMPLE:
        sampled[key] = pd.read_parquet(path)
        print(f"{key}: read {path.name} (no Earth Engine compute used)")
        continue
    print(f"{key}: reading {len(entry['years'])} year(s) at {len(points):,} points")
    try:
        table, run_info = sample_product(key, entry)
    except RuntimeError as error:  # cost limit
        stopped[key] = str(error)
        print(f"{key}: not read, {error}")
        continue
    except ee.EEException as error:
        failures[key] = str(error)
        print(f"{key}: Earth Engine error, skipped: {error}")
        continue
    table.to_parquet(path, index=False)
    path.with_suffix(".run.json").write_text(json.dumps(run_info, indent=2))
    sampled[key] = table
    print(f"{key}: saved {path.name}: {run_info}")

print("Not read because of the cost limit:", list(stopped) or "none")
print("Earth Engine errors:", failures or "none")
run_infos = [json.loads(product_parquet(key).with_suffix(".run.json").read_text()) for key in sampled
             if product_parquet(key).with_suffix(".run.json").exists()]
total_eecu_hours = sum(info["eecu_hours"] for info in run_infos)
print(f"Compute used to read the {len(sampled)} products: {total_eecu_hours:.2f} EECU-hours "
      f"({100 * total_eecu_hours / 150:.1f} % of the Community tier's 150 EECU-hours / month)")

hansen: read hansen_h3res7.parquet (no Earth Engine compute used)
tmf: read tmf_h3res7.parquet (no Earth Engine compute used)
gfc2020: read gfc2020_h3res7.parquet (no Earth Engine compute used)
esri: read esri_h3res7.parquet (no Earth Engine compute used)
dynamic_world: read dynamic_world_h3res7.parquet (no Earth Engine compute used)
worldcover: read worldcover_h3res7.parquet (no Earth Engine compute used)
glc_fcs30d: read glc_fcs30d_h3res7.parquet (no Earth Engine compute used)
glad_glclu: read glad_glclu_h3res7.parquet (no Earth Engine compute used)
cgls_lc100: read cgls_lc100_h3res7.parquet (no Earth Engine compute used)
modis: read modis_h3res7.parquet (no Earth Engine compute used)
Not read because of the cost limit: none
Earth Engine errors: none
Compute used to read the 10 products: 29.89 EECU-hours (19.9 % of the Community tier's 150 EECU-hours / month)


## 7. Defining class codes, forest and the common legend

`samples` has one row per point, product and year:

| Column | Meaning |
|---|---|
| `native` | the product's own class code (Hansen: 1 = forest, 0 = not) |
| `forest` | 1 forest, 0 not forest, NaN no data (cloud, gap) |
| `lulc` | class in the common legend (0 = no data, or forest-only product) |

In [10]:
def to_long(key, wide):
    """One row per point and year: native code, forest (1/0, NaN = no data) and harmonised class (0 = none)."""
    spec = PRODUCTS[key]
    if key == "hansen":
        tree_cover, loss_year, datamask = wide["treecover2000"], wide["lossyear"], wide["datamask"]
        frames = []
        for year in available["hansen"]["years"]:
            lost = (loss_year > 0) & (loss_year <= year - 2000)
            forest = ((tree_cover > HANSEN_CANOPY_THRESHOLD) & ~lost).astype(float)
            frames.append(pd.DataFrame({
                "point_id": wide["point_id"], "year": year, "native": forest.astype(int),
                "forest": forest.where(datamask != NODATA), "lulc": 0,
            }))
        long = pd.concat(frames, ignore_index=True)
    else:
        long = wide.melt(id_vars="point_id", var_name="year", value_name="native")
        long["year"] = long["year"].astype(int)
        no_data = long["native"].isin([NODATA, *spec["nodata"]])
        long["forest"] = long["native"].isin(spec["forest_classes"]).astype(float).mask(no_data)
        crosswalk = spec.get("crosswalk")
        long["lulc"] = long["native"].map(crosswalk).fillna(0).astype(int) if crosswalk else 0
    long["product"] = key
    return long


# Codes found at the points but missing from a crosswalk: check them against the product's legend
for key, wide in sampled.items():
    spec = PRODUCTS[key]
    if "crosswalk" in spec:
        codes = set(pd.unique(wide.drop(columns="point_id").to_numpy().ravel()).tolist())
        unknown = sorted(codes - set(spec["crosswalk"]) - set(spec["nodata"]) - {NODATA})
        if unknown:
            print(f"{PRODUCT_LABELS[key]}: codes not in the crosswalk (counted as no class): {unknown}")

samples = pd.concat([to_long(key, wide) for key, wide in sampled.items()], ignore_index=True)
samples = samples.merge(points[["point_id", "weight_km2", "ECO_NAME"]], on="point_id")

# Point × year tables per product, used by every analysis below
forest_wide = {key: samples[samples["product"] == key].pivot(index="point_id", columns="year", values="forest")
               for key in sampled}
lulc_wide = {key: samples[samples["product"] == key].pivot(index="point_id", columns="year", values="lulc")
             for key in sampled if PRODUCTS[key]["kind"] == "lulc"}
print(f"{len(samples):,} rows: {len(sampled)} products, {samples.groupby('product')['year'].nunique().sum()} product-years")

6,918,864 rows: 10 products, 122 product-years


## 8. Results

`area_estimate` turns a 0/1 value at the points into an area and its standard error. Missing values count as 0 (not
in the class).

In [11]:
PLOT_TEMPLATE = "plotly_white"


def area_estimate(indicator, point_weights):
    """Area (km²) of the points where `indicator` is 1, and its standard error. Missing values count as 0."""
    indicator = indicator.fillna(0).astype(float)
    point_weights = point_weights.reindex(indicator.index)
    share = np.average(indicator, weights=point_weights)
    total = point_weights.sum()
    return total * share, total * np.sqrt(share * (1 - share) / len(indicator))


def mark_method_change(fig):
    fig.add_vline(x=METHOD_CHANGE_YEAR - 0.5, line_dash="dash", line_color="grey")
    fig.add_annotation(x=METHOD_CHANGE_YEAR - 0.5, y=1, yref="paper", text="Hansen method change", showarrow=False,
                       xanchor="left", yanchor="bottom", font={"color": "grey"})
    return fig

### 8.1 Sampling method check

This section tests this sampling method: reading products at 57,000 points and check if these points are enough.

Hansen has been counted by `deforestation_hansen.ipynb` with all ~320 million Hansen pixels in Guinea. Here Hansen's forest area and annual loss are estimated from our points, and compared with that full count.

- If the point estimates are within 2 standard errors of the full count for about 95 % of the rows, the points
  reproduce a map's own numbers correctly. Since **every product is read at the same points**, this holds for all
  of them, whatever their quality.
- If not, the sample is too small or something is wrong in the sampling, and nothing below should be trusted.

In [12]:
if "hansen" in forest_wide and HANSEN_LOSS_PARQUET.exists():
    wall = pd.read_parquet(HANSEN_LOSS_PARQUET)
    wall_column = f"forest2000_gt{HANSEN_CANOPY_THRESHOLD}_km2"
    hansen_forest = forest_wide["hansen"]
    km2, se = area_estimate(hansen_forest[2000], weights)
    rows = [{"quantity": "Forest 2000", "wall_to_wall_km2": wall[wall_column].sum(), "sample_km2": km2, "se_km2": se}]
    for year in sorted(wall.loc[wall["lossyear"] > 0, "year"].unique()):
        lost = ((hansen_forest[year - 1] == 1) & (hansen_forest[year] == 0)).astype(float)
        km2, se = area_estimate(lost, weights)
        rows.append({"quantity": f"Loss {year}", "wall_to_wall_km2": wall.loc[wall["year"] == year, wall_column].sum(),
                     "sample_km2": km2, "se_km2": se})
    sample_check = pd.DataFrame(rows)
    sample_check["difference_in_se"] = (sample_check["sample_km2"] - sample_check["wall_to_wall_km2"]) / sample_check["se_km2"]
    within = (sample_check["difference_in_se"].abs() <= 2).mean()
    print(f"{within:.0%} of the rows are within 2 standard errors of the wall-to-wall value (expected: ~95 %)")
    display(sample_check.round(1))
else:
    print("Needs the Hansen sample and hansen_loss_by_polygon_cell_year.parquet (deforestation_hansen.ipynb)")

100% of the rows are within 2 standard errors of the wall-to-wall value (expected: ~95 %)


,quantity,wall_to_wall_km2,sample_km2,se_km2,difference_in_se
0,Forest 2000,81725.5,81527.3,485.2,-0.4
1,Loss 2001,186.0,139.3,24.5,-1.9
2,Loss 2002,291.2,293.5,35.6,0.1
3,Loss 2003,91.8,79.8,18.6,-0.6
4,Loss 2004,173.2,207.2,29.9,1.1
5,Loss 2005,148.2,140.2,24.6,-0.3
6,Loss 2006,154.9,161.2,26.4,0.2
7,Loss 2007,343.5,372.2,40.1,0.7
8,Loss 2008,218.1,201.3,29.5,-0.6
9,Loss 2009,241.3,217.1,30.6,-0.8


### 8.2 Forest area per product and year

Error bars are ± 1 standard error. The products use different forest definitions (section 3), so different levels are
expected; what matters for deforestation is mostly the **trend** and its smoothness.

- The black dashed line is the official forest area reported to FAO (FRA 2020 country report for Guinea: 6.19 million ha in 2020). 
- Use it with care: the report states that no forest inventory is more recent than 20 years, and that the series reproduces trends from a 1988 vegetation map (FAO 2020).

In [13]:
rows = []
for key, wide in forest_wide.items():
    for year in wide.columns:
        km2, se = area_estimate(wide[year], weights)
        rows.append({"product": key, "label": PRODUCT_LABELS[key], "year": year, "forest_km2": km2, "se_km2": se,
                     "forest_%_of_guinea": 100 * km2 / TOTAL_KM2, "no_data_%": 100 * wide[year].isna().mean()})
forest_area = pd.DataFrame(rows)

fig = px.line(
    forest_area, x="year", y="forest_km2", error_y="se_km2", color="label", markers=True,
    color_discrete_map=LABEL_COLORS, custom_data=["forest_%_of_guinea", "no_data_%"],
    labels={"forest_km2": "Forest (km²)", "year": "Year", "label": "Product"},
    title="Forest area in Guinea according to each product", template=PLOT_TEMPLATE,
)
fig.update_traces(hovertemplate="%{x}: %{y:,.0f} km² (%{customdata[0]:.1f} % of Guinea, %{customdata[1]:.1f} % no data)")
# Official forest area, FAO FRA 2020 country report for Guinea (table 1a, 1,000 ha), extrapolated from 1988 data
FRA_2020_FOREST_KHA = {2000: 6929, 2010: 6569, 2015: 6389, 2016: 6349, 2017: 6309, 2018: 6269, 2019: 6229, 2020: 6189}
fig.add_trace(go.Scatter(x=list(FRA_2020_FOREST_KHA), y=[10 * kha for kha in FRA_2020_FOREST_KHA.values()],
                         mode="lines", line={"color": "black", "dash": "dash"}, name="FAO FRA 2020 (official)",
                         hovertemplate="%{x}: %{y:,.0f} km² (FAO FRA 2020)<extra></extra>"))
mark_method_change(fig).show()

### 8.3 Forest in the comparison year, by ecoregion

Each product at `COMPARISON_YEAR`, or its closest year. 

JRC TMF only maps moist forest, so it is expected to be near zero in the dry ecoregions.

In [14]:
def nearest_year(years, target, max_gap):
    candidates = [year for year in years if abs(year - target) <= max_gap]
    return min(candidates, key=lambda year: (abs(year - target), -year)) if candidates else None


comparison_years = {key: nearest_year(list(wide.columns), COMPARISON_YEAR, MAX_YEAR_GAP) for key, wide in forest_wide.items()}
comparison_years = {key: year for key, year in comparison_years.items() if year is not None}
print("Year used per product:", {PRODUCT_LABELS[key]: year for key, year in comparison_years.items()})

snapshot = pd.DataFrame({key: forest_wide[key][year] for key, year in comparison_years.items()})
snapshot_weights = weights.reindex(snapshot.index)
ecoregion_of_point = points.set_index("point_id")["ECO_NAME"].reindex(snapshot.index)

forest_by_ecoregion = snapshot.fillna(0).mul(snapshot_weights, axis=0).groupby(ecoregion_of_point).sum()
ecoregion_km2 = snapshot_weights.groupby(ecoregion_of_point).sum()
forest_share_by_ecoregion = 100 * forest_by_ecoregion.div(ecoregion_km2, axis=0)

plot_table = forest_share_by_ecoregion.rename(columns=PRODUCT_LABELS).reset_index().melt(
    id_vars="ECO_NAME", var_name="product", value_name="forest_%")
fig = px.bar(
    plot_table, x="ECO_NAME", y="forest_%", color="product", barmode="group", color_discrete_map=LABEL_COLORS,
    labels={"forest_%": "Forest (% of the ecoregion)", "ECO_NAME": "", "product": "Product"},
    title=f"Forest share per ecoregion, around {COMPARISON_YEAR}", template=PLOT_TEMPLATE,
)
fig.update_traces(hovertemplate="%{y:.1f} %")
fig.show()
forest_by_ecoregion.rename(columns=PRODUCT_LABELS).round(0)

Year used per product: {'Hansen GFC': 2020, 'JRC TMF': 2020, 'JRC GFC2020': 2020, 'Esri LULC': 2020, 'Dynamic World': 2020, 'ESA WorldCover': 2020, 'GLC_FCS30D': 2020, 'GLAD GLCLU': 2020, 'Copernicus LC100': 2019, 'MODIS MCD12Q1': 2020}


,Hansen GFC,JRC TMF,JRC GFC2020,Esri LULC,Dynamic World,ESA WorldCover,GLC_FCS30D,GLAD GLCLU,Copernicus LC100,MODIS MCD12Q1
ECO_NAME,,,,,,,,,,
(outside the ecoregion polygons),5.0,5.0,9.0,0.0,0.0,13.0,9.0,0.0,0.0,0.0
Guinean forest-savanna,40066.0,1987.0,67470.0,80239.0,88703.0,59995.0,127438.0,109189.0,101929.0,801.0
Guinean mangroves,1122.0,560.0,1268.0,1502.0,1779.0,1952.0,2362.0,1460.0,1907.0,28.0
Guinean montane forests,7824.0,1152.0,12394.0,13511.0,16084.0,11585.0,24766.0,21337.0,19728.0,669.0
West Sudanian savanna,2360.0,33.0,37220.0,27543.0,27571.0,16074.0,44838.0,45826.0,39144.0,21.0
Western Guinean lowland forests,13072.0,5503.0,6179.0,16066.0,17712.0,14623.0,20944.0,18237.0,18819.0,1375.0


### 8.4 Pairwise agreement on forest

For each pair of products, only points where both have data:

- **Agreement**: share of the area where both say forest or both say non-forest. Can be inflated by the large area both call non-forest.
- **Forest overlap (IoU)**: area both call forest / area at least one calls forest. 
- 100 % means the same forest map, 50 % means half of the forest of the two maps is not shared.

Sources: overall agreement is the basic measure of map comparison (Congalton 1991); the forest overlap is the
Jaccard index (Jaccard 1912). Pontius & Millones (2011) recommend these simple proportions over kappa.

In [15]:
keys = list(snapshot.columns)
agreement = pd.DataFrame(index=keys, columns=keys, dtype=float)
forest_iou = pd.DataFrame(index=keys, columns=keys, dtype=float)
for a in keys:
    for b in keys:
        valid = snapshot[a].notna() & snapshot[b].notna()
        forest_a, forest_b, w = snapshot.loc[valid, a] == 1, snapshot.loc[valid, b] == 1, snapshot_weights[valid]
        agreement.loc[a, b] = 100 * np.average(forest_a == forest_b, weights=w)
        union = w[forest_a | forest_b].sum()
        forest_iou.loc[a, b] = 100 * w[forest_a & forest_b].sum() / union if union else np.nan

for matrix, title in [(forest_iou, "Forest overlap (IoU, %)"), (agreement, "Agreement forest / non-forest (%)")]:
    fig = px.imshow(
        matrix.rename(index=PRODUCT_LABELS, columns=PRODUCT_LABELS).round(0), text_auto=True,
        color_continuous_scale="Greens", zmin=0, zmax=100, aspect="auto",
        title=f"{title}, around {COMPARISON_YEAR}", template=PLOT_TEMPLATE,
    )
    fig.update_layout(height=550)
    fig.show()

### 8.5 Each product against the majority of the others

For each product, the other products vote at every point (forest if more than half of those with data say forest).

| Column | Meaning |
|---|---|
| `agreement_with_majority_%` | share of the area where the product agrees with the majority of the others |
| `over_mapping_%` | share of the product's forest that the majority calls non-forest |
| `under_mapping_%` | share of the majority's forest that the product calls non-forest |

The vote counts are also used to stratify the reference sample in section 10.

Sources: comparing global land-cover maps with each other to find where they disagree follows Herold et al.
(2008). Over- and under-mapping are the commission and omission errors of Congalton (1991), measured here
against the other products instead of against reference data.

In [16]:
forest_votes = (snapshot == 1).sum(axis=1)
valid_votes = snapshot.notna().sum(axis=1)
point_stratum = pd.Series(
    np.select(
        [valid_votes == 0, forest_votes == valid_votes, forest_votes == 0],
        ["no data", "all forest", "all non-forest"],
        default="products disagree",
    ),
    index=snapshot.index, name="stratum",
)

rows = []
for key in keys:
    others_share = snapshot.drop(columns=key).mean(axis=1)  # share of the other products saying forest
    decided = snapshot[key].notna() & others_share.notna() & (others_share != 0.5)
    product_forest = snapshot.loc[decided, key] == 1
    majority_forest = others_share[decided] > 0.5
    w = snapshot_weights[decided]
    rows.append({
        "product": PRODUCT_LABELS[key],
        "agreement_with_majority_%": 100 * np.average(product_forest == majority_forest, weights=w),
        "over_mapping_%": 100 * w[product_forest & ~majority_forest].sum() / w[product_forest].sum(),
        "under_mapping_%": 100 * w[~product_forest & majority_forest].sum() / w[majority_forest].sum(),
    })
consensus = pd.DataFrame(rows).set_index("product")

area_by_votes = snapshot_weights.groupby(forest_votes).sum().rename("area_km2").rename_axis("products_saying_forest").reset_index()
fig = px.bar(
    area_by_votes, x="products_saying_forest", y="area_km2",
    labels={"products_saying_forest": f"Number of products (of {len(keys)}) calling the point forest", "area_km2": "Area (km²)"},
    title=f"How contested is the forest map? ({COMPARISON_YEAR})", template=PLOT_TEMPLATE,
)
fig.update_traces(marker_color="#2e7d32", hovertemplate="%{x} products: %{y:,.0f} km²")
fig.show()
print(snapshot_weights.groupby(point_stratum).sum().div(TOTAL_KM2).mul(100).round(1).rename("% of Guinea"))
consensus.round(1).sort_values("agreement_with_majority_%", ascending=False)

stratum
all forest            0.8
all non-forest        5.3
products disagree    94.0
Name: % of Guinea, dtype: float64


,agreement_with_majority_%,over_mapping_%,under_mapping_%
product,,,
Esri LULC,83.8,18.4,11.1
Dynamic World,81.2,23.9,7.8
ESA WorldCover,76.1,12.3,33.4
JRC GFC2020,71.7,25.2,29.1
Copernicus LC100,68.0,37.6,8.3
GLAD GLCLU,65.7,40.9,3.2
Hansen GFC,63.0,10.9,59.3
GLC_FCS30D,57.6,46.8,0.8
JRC TMF,44.1,10.0,94.2


#### Where do the products disagree?

Share of each hexagon (resolution `MAP_H3_RESOLUTION`) where the products do not all agree on forest / non-forest.
The disagreement comes mostly from product's definitions difference.

In [17]:
def h3_geojson(h3_ids_to_draw):
    """Lightweight GeoJSON for plotly maps: only the outline of each hexagon, coordinates rounded to ~10 m."""
    features = []
    for h3_id in h3_ids_to_draw:
        ring = [[round(lng, 4), round(lat, 4)] for lat, lng in h3.cell_to_boundary(h3_id)]
        features.append({"type": "Feature", "id": h3_id, "geometry": {"type": "Polygon", "coordinates": [ring + ring[:1]]}})
    return {"type": "FeatureCollection", "features": features}


parent_of_point = points.set_index("point_id")[PARENT_COLUMN].reindex(snapshot.index)
hex_weight = snapshot_weights.groupby(parent_of_point).sum()
disagreement_map = pd.DataFrame({
    "h3_id": hex_weight.index,
    "disagreement_%": 100 * (snapshot_weights * (point_stratum == "products disagree")).groupby(parent_of_point).sum() / hex_weight,
    "products_saying_forest_%": 100 * (snapshot_weights * forest_votes / valid_votes.replace(0, np.nan)).groupby(parent_of_point).sum() / hex_weight,
    "main_ecoregion": ecoregion_of_point.groupby(parent_of_point).agg(lambda names: names.mode().iloc[0]),
}).round(1).reset_index(drop=True)

fig = px.choropleth_map(
    disagreement_map, geojson=h3_geojson(disagreement_map["h3_id"]), locations="h3_id", color="disagreement_%",
    color_continuous_scale="Purples", range_color=(0, 100),
    hover_data={"main_ecoregion": True, "products_saying_forest_%": True, "h3_id": False},
    map_style="carto-positron", center={"lat": 10.4, "lon": -11}, zoom=5.6, opacity=0.85,
    labels={"disagreement_%": "Products disagree (% of area)"},
    title=f"Where the forest maps disagree ({COMPARISON_YEAR}), H3 res {MAP_H3_RESOLUTION}",
)
fig.update_traces(marker_line_width=0)
fig.update_layout(height=650, margin={"r": 0, "l": 0, "b": 0})
fig.show()

agreement_by_ecoregion = (
    pd.crosstab(ecoregion_of_point, point_stratum, values=snapshot_weights, aggfunc="sum", normalize="index")
    .mul(100).round(1)
)
agreement_by_ecoregion

stratum,all forest,all non-forest,products disagree
ECO_NAME,,,
(outside the ecoregion polygons),0.0,81.4,18.6
Guinean forest-savanna,0.0,3.8,96.2
Guinean mangroves,0.0,9.4,90.6
Guinean montane forests,2.2,3.1,94.7
West Sudanian savanna,0.0,10.3,89.7
Western Guinean lowland forests,5.8,2.7,91.5


### 8.6 Land-cover products

- Forest-only products (Hansen, TMF, GFC2020) are left out. 
- The cross-table shows, for one pair of products, how the area one of them calls each class is called by the other.

In [18]:
CROSSTAB_PAIR = ("esri", "worldcover")

lulc_keys = [key for key in comparison_years if key in lulc_wide]
lulc_snapshot = pd.DataFrame({key: lulc_wide[key][comparison_years[key]] for key in lulc_keys})

composition = (
    pd.DataFrame({key: snapshot_weights.groupby(lulc_snapshot[key]).sum() for key in lulc_keys})
    .drop(index=0, errors="ignore")
    .rename(index=HARMONISED_CLASSES, columns=PRODUCT_LABELS)
    .fillna(0)
)
plot_table = composition.reset_index(names="class").melt(id_vars="class", var_name="product", value_name="km2")
fig = px.bar(
    plot_table, y="product", x="km2", color="class", orientation="h", color_discrete_map=HARMONISED_COLORS,
    labels={"km2": "Area (km²)", "product": "", "class": "Class"},
    title=f"Land cover of Guinea according to each product (around {COMPARISON_YEAR})", template=PLOT_TEMPLATE,
)
fig.update_traces(hovertemplate="%{x:,.0f} km²")
fig.show()

lulc_agreement = pd.DataFrame(index=lulc_keys, columns=lulc_keys, dtype=float)
for a in lulc_keys:
    for b in lulc_keys:
        valid = (lulc_snapshot[a] > 0) & (lulc_snapshot[b] > 0)
        lulc_agreement.loc[a, b] = 100 * np.average(lulc_snapshot.loc[valid, a] == lulc_snapshot.loc[valid, b],
                                                     weights=snapshot_weights[valid])
fig = px.imshow(
    lulc_agreement.rename(index=PRODUCT_LABELS, columns=PRODUCT_LABELS).round(0), text_auto=True,
    color_continuous_scale="Blues", zmin=0, zmax=100, aspect="auto",
    title=f"Agreement on the 7 common classes (%), around {COMPARISON_YEAR}", template=PLOT_TEMPLATE,
)
fig.update_layout(height=500)
fig.show()

a, b = CROSSTAB_PAIR
if a in lulc_snapshot and b in lulc_snapshot:
    crosstab = pd.crosstab(
        lulc_snapshot[a].map(HARMONISED_CLASSES).rename(PRODUCT_LABELS[a]),
        lulc_snapshot[b].map(HARMONISED_CLASSES).rename(PRODUCT_LABELS[b]),
        values=snapshot_weights, aggfunc="sum",
    ).fillna(0)
    print(f"km²: rows = {PRODUCT_LABELS[a]}, columns = {PRODUCT_LABELS[b]}")
    display(crosstab.round(0))

km²: rows = Esri LULC, columns = ESA WorldCover


ESA WorldCover,Bare / sparse vegetation,Built-up,Cropland,Herbaceous wetland,Shrub & grassland,Tree cover,Water
Esri LULC,,,,,,,
Bare / sparse vegetation,13.0,0.0,0.0,5.0,0.0,0.0,60.0
Built-up,242.0,754.0,176.0,4.0,1086.0,384.0,8.0
Cropland,27.0,4.0,861.0,92.0,1269.0,224.0,0.0
Herbaceous wetland,4.0,0.0,57.0,93.0,30.0,1554.0,5.0
Shrub & grassland,995.0,126.0,4238.0,257.0,81934.0,10742.0,40.0
Tree cover,51.0,30.0,23.0,4.0,47631.0,91118.0,4.0
Water,70.0,0.0,13.0,31.0,48.0,221.0,789.0


### 8.7 Temporal consistency

For every product with at least three consecutive years, each point and year is classified as:

- **loss**: forest the year before, non-forest this year;
- **gain**: non-forest the year before, forest this year;
- **persistent loss**: loss that is still non-forest the next year;
- **flip**: the class changes for one year only and comes back (forest → non-forest → forest, or the reverse).
  Real clearing does not grow back into forest within a year, so flips are mostly classification noise.

A product made the same way every year should have few flips, and much less gain than loss over a few years.
`flips_per_loss` > 1 means a year's "loss" is more likely noise than clearing.

The summary uses only the years inside `COMMON_WINDOW`, so all products are compared on the same years.

Sources: classification errors in maps of different years create spurious change (Pontius & Lippitt 2006);
one-year reversals are the typical symptom in annual land-cover series, which is why MODIS land cover is
post-processed to remove them (Abercrombie & Friedl 2016). Requiring a loss to persist follows the logic of
JRC TMF, where a disturbance only counts as deforestation if it lasts (Vancutsem et al. 2021).

In [19]:
def change_table(key):
    """Loss, gain, persistent loss and flips per year (km²) from the point × year forest table of a product."""
    wide = forest_wide[key]
    rows = []
    for year in wide.columns:
        if year - 1 not in wide.columns:
            continue
        before, now = wide[year - 1], wide[year]
        loss = (before == 1) & (now == 0)
        row = {"product": key, "label": PRODUCT_LABELS[key], "year": year}
        row["loss_km2"], row["loss_se_km2"] = area_estimate(loss.astype(float), weights)
        row["gain_km2"], _ = area_estimate(((before == 0) & (now == 1)).astype(float), weights)
        if year + 1 in wide.columns:
            after = wide[year + 1]
            persistent = loss & (after == 0)
            flip = before.notna() & now.notna() & after.notna() & (before == after) & (now != before)
            row["persistent_loss_km2"], row["persistent_loss_se_km2"] = area_estimate(persistent.astype(float), weights)
            row["flip_km2"], _ = area_estimate(flip.astype(float), weights)
        rows.append(row)
    return pd.DataFrame(rows)


annual_keys = [
    key for key, wide in forest_wide.items()
    if len(wide.columns) >= 3 and all(year + 1 in wide.columns for year in sorted(wide.columns)[:-1])
]
changes = pd.concat([change_table(key) for key in annual_keys], ignore_index=True)

in_window = changes[changes["year"].between(*COMMON_WINDOW)]
consistency = in_window.groupby("label")[["loss_km2", "persistent_loss_km2", "gain_km2", "flip_km2"]].mean()
consistency.columns = [f"mean_{column}_per_year" for column in consistency.columns]
consistency["flip_%_of_guinea_per_year"] = 100 * consistency["mean_flip_km2_per_year"] / TOTAL_KM2
consistency["gain_per_loss"] = consistency["mean_gain_km2_per_year"] / consistency["mean_loss_km2_per_year"]
consistency["flips_per_loss"] = consistency["mean_flip_km2_per_year"] / consistency["mean_loss_km2_per_year"]
print(f"Means over {COMMON_WINDOW[0]}–{COMMON_WINDOW[1]}")

fig = px.bar(
    consistency.reset_index(), x="label", y="flip_%_of_guinea_per_year", color="label", color_discrete_map=LABEL_COLORS,
    labels={"flip_%_of_guinea_per_year": "One-year flips (% of Guinea per year)", "label": ""},
    title=f"Classification noise: one-year forest flips, {COMMON_WINDOW[0]}–{COMMON_WINDOW[1]}", template=PLOT_TEMPLATE,
)
fig.update_layout(showlegend=False)
fig.update_traces(hovertemplate="%{y:.2f} % per year")
fig.show()
consistency.round(2)

Means over 2017–2022


,mean_loss_km2_per_year,mean_persistent_loss_km2_per_year,mean_gain_km2_per_year,mean_flip_km2_per_year,flip_%_of_guinea_per_year,gain_per_loss,flips_per_loss
label,,,,,,,
Copernicus LC100,54.79,64.50,1.49,0.00,0.00,0.03,0.00
Esri LULC,15712.87,10349.98,9876.58,8364.54,3.41,0.63,0.53
GLC_FCS30D,1170.48,1156.28,683.90,283.62,0.12,0.58,0.24
Hansen GFC,1792.63,1792.63,0.00,0.00,0.00,0.00,0.00
JRC TMF,96.06,96.06,510.14,0.00,0.00,5.31,0.00
MODIS MCD12Q1,353.20,289.11,350.74,182.73,0.07,0.99,0.52


### 8.8 Annual forest loss per product

- Keeps only **persistent** loss, which removes most of the flip noise. 
- Error bars are ± 1 standard error. 
- The last year of each product has no persistent loss (no next year yet).

**Official benchmark (black).** Guinea's REDD+ reference level submitted to the UNFCCC in 2026 (NERF,
period 2016–2020, under technical assessment) gives the official annual deforestation for 2016–2024, from
1,688 sample plots interpreted in Collect Earth Online with the national forest definition (Code forestier
2017). Its error bars are 95 % confidence intervals and they are wide (56–150 % of the value), so it tells the
**order of magnitude** a product should match, not the exact year-to-year shape. Note the NERF separates
*deforestation* (forest → non-forest, ~640 km²/yr on average) from *degradation* (dense → open forest,
~1,950 km²/yr), which products like Hansen partly count as loss.

The table compares each product's mean annual loss with the official mean over the same years.

In [20]:
# Official annual deforestation, Guinea NERF 2026 (Tableau 8): year -> (hectares, half-width of the 95 % CI in hectares)
NERF_DEFORESTATION_HA = {
    2016: (99_057.23, 72_509.93), 2017: (22_908.00, 34_392.25), 2018: (104_242.07, 71_416.86),
    2019: (64_695.23, 54_299.52), 2020: (22_908.00, 31_778.74), 2021: (30_333.22, 36_270.91),
    2022: (62_454.85, 51_174.91), 2023: (125_361.67, 69_900.66), 2024: (41_787.22, 39_425.71),
}
nerf = pd.DataFrame([{"year": year, "loss_km2": ha / 100, "ci95_km2": ci / 100}
                     for year, (ha, ci) in NERF_DEFORESTATION_HA.items()])

for column, se_column, title in [
    ("loss_km2", "loss_se_km2", "Annual forest loss (forest → non-forest)"),
    ("persistent_loss_km2", "persistent_loss_se_km2", "Annual persistent forest loss (still non-forest the next year)"),
]:
    fig = px.line(
        changes.dropna(subset=[column]), x="year", y=column, error_y=se_column, color="label", markers=True,
        color_discrete_map=LABEL_COLORS, labels={column: "Loss (km²)", "year": "Year", "label": "Product"},
        title=title, template=PLOT_TEMPLATE,
    )
    fig.update_traces(hovertemplate="%{x}: %{y:,.0f} km²")
    fig.add_trace(go.Scatter(x=nerf["year"], y=nerf["loss_km2"], error_y={"array": nerf["ci95_km2"]}, mode="markers",
                             marker={"color": "black", "size": 9}, name="Official deforestation (NERF, 95 % CI)",
                             hovertemplate="%{x}: %{y:,.0f} km² (NERF)<extra></extra>"))
    mark_method_change(fig).show()

rows = []
for label, product_changes in changes[changes["year"].isin(nerf["year"])].groupby("label"):
    years = product_changes["year"]
    official = nerf.loc[nerf["year"].isin(years), "loss_km2"].mean()
    rows.append({"product": label, "years": f"{years.min()}–{years.max()}",
                 "mean_loss_km2": product_changes["loss_km2"].mean(),
                 "mean_persistent_loss_km2": product_changes["persistent_loss_km2"].mean(),
                 "official_deforestation_km2": official,
                 "persistent_loss_vs_official": product_changes["persistent_loss_km2"].mean() / official})
official_comparison = pd.DataFrame(rows).set_index("product")
official_comparison.round(2)

,years,mean_loss_km2,mean_persistent_loss_km2,official_deforestation_km2,persistent_loss_vs_official
product,,,,,
Copernicus LC100,2016–2019,58.77,66.58,727.26,0.09
Esri LULC,2018–2024,14826.66,9305.52,645.40,14.42
GLC_FCS30D,2016–2022,1189.84,1154.06,580.86,1.99
Hansen GFC,2016–2024,1769.97,1769.97,637.50,2.78
JRC TMF,2016–2024,117.77,117.77,637.50,0.18
MODIS MCD12Q1,2016–2024,396.32,316.21,637.50,0.50


### 8.9 Accounting on Forest Definition Difference

Guinea's legal definition counts as forest any land with more than 10 % tree canopy (Code forestier 2017, used by the NERF). 
The values already read are classified again with other thresholds: Hansen with canopy > 10, 15 and 30 % in 2000, and MODIS with or without its savanna classes (IGBP 8, woody savanna, 30–60 % canopy; IGBP 9, savanna, 10–30 % canopy; Friedl et al. 2010).

In [21]:
hansen_values = sampled["hansen"].set_index("point_id")
tree_cover, loss_year = hansen_values["treecover2000"], hansen_values["lossyear"]
modis_values = sampled["modis"].set_index("point_id")[str(comparison_years["modis"])]
loss_years = [year for year in NERF_DEFORESTATION_HA if year - 2000 <= loss_year.max()]

rows = []
for threshold in (10, 15, 30):
    forest_2000 = tree_cover > threshold
    lost = (loss_year > 0) & (loss_year <= COMPARISON_YEAR - 2000)
    annual_loss = [area_estimate(forest_2000 & (loss_year == year - 2000), weights)[0] for year in loss_years]
    rows.append({"definition": f"Hansen, canopy > {threshold} % in 2000",
                 "forest_km2": area_estimate(forest_2000 & ~lost, weights)[0], "mean_loss_km2_per_year": np.mean(annual_loss)})
for classes, name in [([1, 2, 3, 4, 5], "forests (> 60 % canopy)"), ([1, 2, 3, 4, 5, 8], "+ woody savanna (> 30 %)"),
                      ([1, 2, 3, 4, 5, 8, 9], "+ savanna (> 10 %)")]:
    rows.append({"definition": f"MODIS, IGBP {name}", "forest_km2": area_estimate(modis_values.isin(classes), weights)[0]})
rows.append({"definition": "Official: FAO FRA 2020 (forest), NERF 2026 (deforestation)",
             "forest_km2": 10 * FRA_2020_FOREST_KHA[2020],
             "mean_loss_km2_per_year": np.mean([NERF_DEFORESTATION_HA[year][0] / 100 for year in loss_years])})
definition_effect = pd.DataFrame(rows).set_index("definition")
definition_effect.insert(1, "forest_%_of_guinea", 100 * definition_effect["forest_km2"] / TOTAL_KM2)
print(f"Forest in {COMPARISON_YEAR}; mean annual loss {loss_years[0]}–{loss_years[-1]}")
definition_effect.round(1)


Forest in 2020; mean annual loss 2016–2024


,forest_km2,forest_%_of_guinea,mean_loss_km2_per_year
definition,,,
"Hansen, canopy > 10 % in 2000",199846.3,81.5,2846.6
"Hansen, canopy > 15 % in 2000",169144.9,68.9,2788.3
"Hansen, canopy > 30 % in 2000",64448.9,26.3,1770.0
"MODIS, IGBP forests (> 60 % canopy)",2892.2,1.2,NaN
"MODIS, IGBP + woody savanna (> 30 %)",15886.8,6.5,NaN
"MODIS, IGBP + savanna (> 10 %)",197982.8,80.7,NaN
"Official: FAO FRA 2020 (forest), NERF 2026 (deforestation)",61890.0,25.2,637.5


## 9. Visual check

The forest map of every product for its comparison year, over satellite imagery.

In [22]:
def forest_mask(key, year):
    """Earth Engine image: 1 where the product says forest in `year`."""
    entry = available[key]
    image = build_image(key, entry, [year])
    if key == "hansen":
        loss_year = image.select("lossyear")
        lost = loss_year.gt(0).And(loss_year.lte(year - 2000))
        return image.select("treecover2000").gt(HANSEN_CANOPY_THRESHOLD).And(lost.Not())
    forest_classes = PRODUCTS[key]["forest_classes"]
    return image.select(str(year)).remap(forest_classes, [1] * len(forest_classes), 0)


m = geemap.Map(center=[10.4, -11], zoom=7)
m.add_basemap("SATELLITE")
for key, year in comparison_years.items():
    m.add_layer(forest_mask(key, year).selfMask().clip(guinea_ee), {"palette": [PRODUCT_COLORS[key]]},
                f"{PRODUCT_LABELS[key]} forest {year}", False)
m.add_layer(ee.FeatureCollection([ee.Feature(guinea_ee)]).style(color="white", fillColor="00000000", width=2), {}, "Guinea")
m

Map(center=[10.4, -11], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

## 10. Note

1. **Availability.** All 10 products are on Earth Engine and were read at the 56,712 points for 29.9 EECU-hours
   (20 % of a month of the Community tier); re-running now costs nothing. For an **annual** series at ≤ 30 m, three
   products qualify: **Hansen** (consistent only 2013–2025), **JRC TMF** (2000–2025, moist forest only) and
   **GLC_FCS30D** (2000–2022, not updated since). Esri is annual at 10 m but too unstable for change (point 5). JRC
   GFC2020, WorldCover and GLAD GLCLU are single or 5-yearly dates: a 2020 baseline, not annual change. Copernicus
   (100 m, ends 2019) and MODIS (500 m) are too coarse for clearings of a few hectares. Dynamic World can give any
   year since 2016 but costs ~7 EECU-hours per year here.
2. **The sampling works** (8.1): the points reproduce Hansen's full pixel count within 2 standard errors for every
   year.
3. **Forest area depends on the definition more than on the product.** Around 2020 the products range from 2,900 km²
   (MODIS, dense forest only) to 220,000 km² (GLC_FCS30D, open forest included): 1 % to 90 % of Guinea. The legal
   threshold (> 10 % canopy) includes most wooded savanna: Hansen gives ~200,000 km² at > 10 % and ~64,000 km² at
   > 30 % (8.9). FAO FRA 2020 (61,900 km²) is close to the 30 % version, but it is extrapolated from a 1988 map.
4. **The products agree almost nowhere.** All ten agree on 6 % of Guinea only (8.5). This is mostly the definition
   effect (TMF maps only moist forest, MODIS only dense forest), so it does not tell which product is right.
5. **Only some products are stable enough for annual change** (8.7). Esri's forest flips for a single year on 3.4 % of
   Guinea every year, and its persistent loss is 14 × the official deforestation: not usable for change. GLC_FCS30D
   (0.12 % flips) and MODIS are stable. Hansen and TMF cannot flip by construction.
6. **No product matches the official deforestation** (NERF, ~640 km²/yr over 2016–2024, 8.8). Hansen gives
   1,770 km²/yr at > 30 % (× 2.8) and ~2,850 at > 10 % (× 4.5), GLC_FCS30D 1,150 (× 2.0), TMF 120 (× 0.2, moist forest
   only). Hansen's loss is closer to the NERF's deforestation **plus** degradation (~2,600 km²/yr): it counts canopy
   loss, not only conversion to non-forest.
7. **Accuracy is not known yet.** Need reference points are labelled (if needed). Nothing above ranks the products.

### 10.1 Decisions

1. **Operational forest definition.** The legal > 10 % canopy (as in the NERF: dense, open and montane forest, NERF
   strata 1–3), or a stricter threshold reported next to it? Forest area and loss change by a factor of ~3 with this
   choice (8.9).
2. **What "deforestation" means in the outputs.** Forest → non-forest only, as in the NERF (degradation reported
   separately), or tree cover loss as in Hansen (includes degradation, fire, fallow clearing).
3. **Candidates.** Hansen (2013–2025 only), JRC TMF (moist south), GLC_FCS30D (whole country, to 2022),
   and our own classification of the HLS mosaics (`satellite_mosaic.ipynb`) if none of them fits the definition.

## References

- Abercrombie, S. P., & Friedl, M. A. (2016). Improving the consistency of multitemporal land cover maps using a
  hidden Markov model. *IEEE Transactions on Geoscience and Remote Sensing*, 54(2), 703–713.
- Congalton, R. G. (1991). A review of assessing the accuracy of classifications of remotely sensed data. *Remote
  Sensing of Environment*, 37(1), 35–46.
- Di Gregorio, A. (2005). *Land Cover Classification System (LCCS), version 2: Classification concepts and user
  manual*. FAO Environment and Natural Resources Service Series 8. Rome: FAO.
- Efron, B., & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall.
- FAO (2020). *Évaluation des ressources forestières mondiales 2020 – Rapport Guinée*. Rome.
  https://openknowledge.fao.org/3/cb0133fr/cb0133fr.pdf
- Friedl, M. A., et al. (2010). MODIS Collection 5 global land cover: Algorithm refinements and characterization
  of new datasets. *Remote Sensing of Environment*, 114(1), 168–182.
- GFOI (2020). *Methods and Guidance from the Global Forest Observations Initiative*, Edition 3.0. Rome: FAO.
- Hansen, M. C., et al. (2013). High-resolution global maps of 21st-century forest cover change. *Science*,
  342(6160), 850–853.
- Herold, M., Mayaux, P., Woodcock, C. E., Baccini, A., & Schmullius, C. (2008). Some challenges in global land cover
  mapping: An assessment of agreement and accuracy in existing 1 km datasets. *Remote Sensing of Environment*,
  112(5), 2538–2556.
- IPCC (2006). *2006 IPCC Guidelines for National Greenhouse Gas Inventories*, Vol. 4, Ch. 3: Consistent
  representation of lands (and its 2019 Refinement).
- Jaccard, P. (1912). The distribution of the flora in the alpine zone. *New Phytologist*, 11(2), 37–50.
- Olofsson, P., Foody, G. M., Herold, M., Stehman, S. V., Woodcock, C. E., & Wulder, M. A. (2014). Good practices
  for estimating area and assessing accuracy of land change. *Remote Sensing of Environment*, 148, 42–57.
- Pontius, R. G., Jr., & Lippitt, C. D. (2006). Can error explain map differences over time? *Cartography and
  Geographic Information Science*, 33(2), 159–171.
- Pontius, R. G., Jr., & Millones, M. (2011). Death to Kappa: birth of quantity disagreement and allocation
  disagreement for accuracy assessment. *International Journal of Remote Sensing*, 32(15), 4407–4429.
- République de Guinée (2017). Loi L/2017/060/AN portant Code forestier.
- République de Guinée (2026). *Niveau de référence des forêts (NRF) et niveau d'émissions de référence des forêts
  (NERF) de la Guinée, période 2016–2020*. Submitted to the UNFCCC.
  https://redd.unfccc.int/media/nerf_de_la_guinee_2026.pdf
- Stehman, S. V. (2009). Sampling designs for accuracy assessment of land cover. *International Journal of Remote
  Sensing*, 30(20), 5243–5272.
- Stehman, S. V. (2014). Estimating area and map accuracy for stratified random sampling when the strata are
  different from the map classes. *International Journal of Remote Sensing*, 35(13), 4923–4939.
- Stehman, S. V., & Foody, G. M. (2019). Key issues in rigorous accuracy assessment of land cover products. *Remote
  Sensing of Environment*, 231, 111199.
- Vancutsem, C., et al. (2021). Long-term (1990–2019) monitoring of forest cover changes in the humid tropics.
  *Science Advances*, 7(10), eabe1603.